In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [2]:
n = 128
tau = 10e-3
dt = 0.5e-3

beta = (1 / n) ** 2
gamma = 1.5 * beta
a_weight = 1

alpha = 0.10315


B0 = 1
seed = 0


def symmetric_kernel(
    n,
    a_weight,
    gamma,
    beta,
):
    idx = np.arange(n)
    d = idx - n // 2
    d = np.where(d > n / 2, d - n, d)
    d = np.where(d < -n / 2, d + n, d)

    sx = d
    r2 = sx**2

    K = a_weight * np.exp(-gamma * r2) - np.exp(-beta * r2)
    K = np.fft.ifftshift(K)
    return K


def asymmetric_kernel(
    n,
):
    idx = np.arange(n)
    d = idx - n // 2
    d = np.where(d > n / 2, d - n, d)
    d = np.where(d < -n / 2, d + n, d)
    K = d * np.exp(-np.abs(d))
    K = np.fft.ifftshift(K)
    return K


sym_kernel = np.fft.fft(symmetric_kernel(n, 1, gamma, beta))
asym_kernel = np.fft.fft(asymmetric_kernel(n))


def recurrent_input(s):
    rec_input = np.zeros_like(s)
    rec_input = np.real(np.fft.ifft(np.fft.fft(s) * sym_kernel))
    return rec_input


def feedforward_input(input):
    return B0 * (1.0) + input * np.real(np.fft.ifft(np.fft.fft(s) * asym_kernel))


def update_neural_state(s, input):
    total_input = recurrent_input(s) + feedforward_input(input)
    return s + (dt / tau) * (-s + np.maximum(total_input, 0.0))

In [3]:

s = np.zeros(n)

s[int(n/2)] = 1

for _ in range(1000):
    s = update_neural_state(s, 0)

initial_state = s.copy()

In [4]:
n_steps = 1400
population_recording = np.zeros((n_steps, n))
for step in range(n_steps):
    s = update_neural_state(s, 0.5)
    population_recording[step] = s.copy()

In [ ]:
fig, ax = plt.subplots()
ax.plot(initial_state, linestyle="dotted", color="black")
ax.set_xlim(0, n - 1)
artists = []
for step in range(n_steps):
    artists.append(ax.plot(population_recording[step], color="black"))

ani = animation.ArtistAnimation(fig=fig, artists=artists, interval=0.0001)
plt.show()

In [ ]:
ani.save(filename="plots/moving_bump.gif", writer="pillow")